### $limitations$


*   The broadcast camera might be closer or further to the players than training dataset which migh cause failures and misdetections, to mitigate that we need a large dataset with multiple broadcast cam distance.
*   occlusion
*   fast moving camera blurring (usually happens at goalkicks)




In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.0 MB/s eta 0:00:00


### $The$ $Model$ $is$ $running$ $at$ $44$ $FPS$

In [ ]:
from ultralytics import YOLO
import os
from IPython.display import HTML
from base64 import b64encode

!gdown -O "0bfacc_0.mp4" "https://drive.google.com/uc?id=12TqauVZ9tLAv8kWxTTBFWtgt2hNQ4_ZF"

model = YOLO('/content/drive/MyDrive/models/football-detection/weights/best.pt')

CONF = float(input("Enter confidence level (0.0-1.0): "))

results = model(
    "0bfacc_0.mp4",
    imgsz=1280,
    conf=CONF,
    save=True,
    project="/content",
    name="video_output"
)

output_dir = "/content/video_output"

output_file = [
    f for f in os.listdir(output_dir)
    if f.endswith(('.mp4', '.avi'))
][0]

output_full_path = os.path.join(output_dir, output_file)

compressed_path = "/content/compressed_output.mp4"

!ffmpeg -y -i "{output_full_path}" -vcodec libx264 -crf 28 "{compressed_path}"

mp4 = open(compressed_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=800 controls>
    <source src="{data_url}" type="video/mp4">
</video>
""")

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import os
import subprocess

# config
CLASS_STYLES = {
    "player":     {"color": (255, 255, 255), "circle": True,  "indicator": "none"},
    "goalkeeper": {"color": (0,   200, 255), "circle": True,  "indicator": "none"},
    "referee":    {"color": (255, 100,   0), "circle": True,  "indicator": "none"},
    "ball":       {"color": (0,   255, 150), "circle": False, "indicator": "triangle"},
}
FONT       = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.35
FONT_THICK = 1

#  drawing helpers
def draw_ellipse(frame, bbox, color, thickness=2):
    """Draw a flat ellipse at the bottom of the bounding box (shadow under feet)."""
    x1, y1, x2, y2 = bbox
    cx = (x1 + x2) // 2
    cy = y2                        # bottom of box
    rx = (x2 - x1) // 2
    ry = max(4, rx // 4)           # flat ellipse
    cv2.ellipse(frame, (cx, cy), (rx, ry), 0, -45, 225, color, thickness, cv2.LINE_AA)

def draw_triangle_indicator(frame, bbox, color, thickness=2):
    """Draw a downward-pointing triangle above the ball."""
    x1, y1, x2, y2 = bbox
    cx  = (x1 + x2) // 2
    tip = y1 - 6                   # tip just above box
    pts = np.array([
        [cx,      tip],
        [cx - 8,  tip - 14],
        [cx + 8,  tip - 14],
    ], dtype=np.int32)
    cv2.polylines(frame, [pts], isClosed=True, color=color, thickness=thickness, lineType=cv2.LINE_AA)
    # small filled dot at tip
    cv2.circle(frame, (cx, tip), 2, color, -1, cv2.LINE_AA)

def draw_label(frame, text, bbox, color):
    """Draw a compact label above the bounding box."""
    x1, y1, x2, y2 = bbox
    cx = (x1 + x2) // 2
    (tw, th), _ = cv2.getTextSize(text, FONT, FONT_SCALE, FONT_THICK)
    tx = cx - tw // 2
    ty = y1 - 18
    # pill background
    pad = 3
    cv2.rectangle(frame,
                  (tx - pad, ty - th - pad),
                  (tx + tw + pad, ty + pad),
                  (0, 0, 0), -1, cv2.LINE_AA)
    cv2.putText(frame, text, (tx, ty), FONT, FONT_SCALE, color, FONT_THICK, cv2.LINE_AA)

def draw_ball_circle(frame, bbox, color, thickness=2):
    """Draw a clean circle around the ball instead of a box."""
    x1, y1, x2, y2 = bbox
    cx = (x1 + x2) // 2
    cy = (y1 + y2) // 2
    r  = max((x2 - x1), (y2 - y1)) // 2 + 4
    cv2.circle(frame, (cx, cy), r, color, thickness, cv2.LINE_AA)

def render_frame(frame, results, model):
    for box in results[0].boxes:
        cls_id    = int(box.cls)
        cls_name  = model.names[cls_id]
        conf      = float(box.conf)
        x1,y1,x2,y2 = map(int, box.xyxy[0])
        bbox      = (x1, y1, x2, y2)
        style     = CLASS_STYLES.get(cls_name, {"color": (200,200,200), "circle": True, "indicator": "none"})
        color     = style["color"]

        if cls_name == "ball":
            draw_ball_circle(frame, bbox, color, thickness=2)
            draw_triangle_indicator(frame, bbox, color, thickness=2)
            draw_label(frame, f"ball {conf:.2f}", bbox, color)
        else:
            draw_ellipse(frame, bbox, color, thickness=2)
            draw_label(frame, f"{cls_name} {conf:.2f}", bbox, color)

    return frame

# download and run
!gdown -O "0bfacc_0.mp4" "https://drive.google.com/uc?id=12TqauVZ9tLAv8kWxTTBFWtgt2hNQ4_ZF"

model      = YOLO('/content/drive/MyDrive/models/football-detection/weights/best.pt')
cap        = cv2.VideoCapture("0bfacc_0.mp4")
fps        = cap.get(cv2.CAP_PROP_FPS)
W          = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H          = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total      = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
raw_out    = "/content/raw_ps_output.mp4"
writer     = cv2.VideoWriter(raw_out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

print(f"Processing {total} frames at {fps:.1f} fps...")
frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    results = model(frame, imgsz=1280, conf=0.3, verbose=False)
    frame   = render_frame(frame, results, model)
    writer.write(frame)
    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f"  {frame_idx}/{total} frames done...")

cap.release()
writer.release()
print("Inference done, compressing...")

# compress
final_out = "/content/ps_style_output.mp4"
subprocess.run([
    "ffmpeg", "-y", "-i", raw_out,
    "-vcodec", "libx264", "-crf", "28",
    "-preset", "fast", "-movflags", "+faststart",
    final_out
], check=True)

original_size   = os.path.getsize(raw_out)   / (1024*1024)
compressed_size = os.path.getsize(final_out) / (1024*1024)
print(f"{original_size:.1f}MB {compressed_size:.1f}MB")

# preview
from IPython.display import HTML
from base64 import b64encode

mp4      = open(final_out, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=800 controls><source src="{data_url}" type="video/mp4"></video>')

In [ ]:
!gdown -O "0bfacc_0.mp4" "https://drive.google.com/uc?id=12TqauVZ9tLAv8kWxTTBFWtgt2hNQ4_ZF"

Downloading...
From: https://drive.google.com/uc?id=12TqauVZ9tLAv8kWxTTBFWtgt2hNQ4_ZF
To: /content/0bfacc_0.mp4
100% 19.9M/19.9M [00:00<00:00, 139MB/s]


In [ ]:
# install dependencies
!pip install supervision transformers umap-learn scikit-learn more-itertools -q
!pip install ultralytics
import cv2, torch, numpy as np, os, subprocess
from tqdm import tqdm
from more_itertools import chunked
from transformers import AutoProcessor, SiglipVisionModel
from sklearn.cluster import KMeans
from ultralytics import YOLO
from PIL import Image
import supervision as sv
import umap
from collections import defaultdict

In [ ]:
#  config
SOURCE_VIDEO   = "0bfacc_0.mp4"
MODEL_PATH     = '/content/drive/MyDrive/models/football-detection/weights/best.pt'
SIGLIP_PATH    = 'google/siglip-base-patch16-224'
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'
STRIDE         = 30
BATCH_SIZE     = 32
CONF           = 0.3

BALL_ID       = 0
GOALKEEPER_ID = 1
PLAYER_ID     = 2
REFEREE_ID    = 3

TEAM_COLORS   = ['#00BFFF', '#FF1493', '#FFD700']
BALL_COLOR    = '#FFD700'

POSSESSION_RADIUS     = 80
POSSESSION_MIN_FRAMES = 3

model = YOLO(MODEL_PATH)

In [ ]:
# collect crops and cluster
print("Step 1/4  Collecting player crops...")
frame_gen = sv.get_video_frames_generator(SOURCE_VIDEO, stride=STRIDE)
crops = []
for frame in tqdm(frame_gen, desc="Crops"):
    results   = model(frame, imgsz=1280, conf=CONF, verbose=False)[0]
    xyxys     = results.boxes.xyxy.cpu().numpy()
    class_ids = results.boxes.cls.cpu().numpy().astype(int)
    for xyxy, cls_id in zip(xyxys, class_ids):
        if cls_id == PLAYER_ID:
            x1,y1,x2,y2 = map(int, xyxy)
            crop = frame[y1:y2, x1:x2]
            if crop.size > 0:
                crops.append(Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)))

print(f"  {len(crops)} player crops collected")

print("Step 2/4  Extracting SigLIP embeddings")
embed_model = SiglipVisionModel.from_pretrained(SIGLIP_PATH).to(DEVICE)
processor   = AutoProcessor.from_pretrained(SIGLIP_PATH)
embeddings_list = []
with torch.no_grad():
    for batch in tqdm(chunked(crops, BATCH_SIZE), desc="Embeddings"):
        inputs  = processor(images=list(batch), return_tensors="pt").to(DEVICE)
        outputs = embed_model(**inputs)
        emb     = torch.mean(outputs.last_hidden_state, dim=1).cpu().numpy()
        embeddings_list.append(emb)
embeddings = np.concatenate(embeddings_list)
print(f"Embeddings shape: {embeddings.shape}")

print("Step 3/4  Clustering into 2 teams")
reducer     = umap.UMAP(n_components=3, random_state=42)
projections = reducer.fit_transform(embeddings)
kmeans      = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans.fit(projections)
print(" Clustering done")

def predict_team(crop_imgs):
    if not crop_imgs:
        return np.array([], dtype=int)
    with torch.no_grad():
        inputs  = processor(images=crop_imgs, return_tensors="pt").to(DEVICE)
        outputs = embed_model(**inputs)
        emb     = torch.mean(outputs.last_hidden_state, dim=1).cpu().numpy()
    proj = reducer.transform(emb)
    return kmeans.predict(proj)

def resolve_goalkeeper_team(players_xyxy, players_team, gk_xyxy):
    if len(gk_xyxy) == 0 or len(players_xyxy) == 0:
        return np.zeros(len(gk_xyxy), dtype=int)
    team_centers = {}
    for t in [0, 1]:
        mask = players_team == t
        if mask.any():
            boxes = players_xyxy[mask]
            cx = (boxes[:,0]+boxes[:,2])/2
            cy = (boxes[:,1]+boxes[:,3])/2
            team_centers[t] = np.array([cx.mean(), cy.mean()])
    results = []
    for xyxy in gk_xyxy:
        cx = (xyxy[0]+xyxy[2])/2
        cy = (xyxy[1]+xyxy[3])/2
        pos = np.array([cx, cy])
        best = min(team_centers, key=lambda t: np.linalg.norm(pos - team_centers[t]))
        results.append(best)
    return np.array(results, dtype=int)

# 3. drawing helpers
FONT       = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.35
FONT_THICK = 1

def hex_to_bgr(h):
    h = h.lstrip('#')
    r,g,b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return (b,g,r)

TEAM_BGR = [hex_to_bgr(c) for c in TEAM_COLORS]
BALL_BGR = hex_to_bgr(BALL_COLOR)

def draw_ellipse(frame, bbox, color, thickness=2):
    x1,y1,x2,y2 = bbox
    cx = (x1+x2)//2; cy = y2
    rx = max(1,(x2-x1)//2); ry = max(4,rx//4)
    cv2.ellipse(frame,(cx,cy),(rx,ry),0,-45,225,color,thickness,cv2.LINE_AA)

def draw_triangle(frame, bbox, color, thickness=2):
    x1,y1,x2,y2 = bbox
    cx = (x1+x2)//2; tip = y1-6
    pts = np.array([[cx,tip],[cx-8,tip-14],[cx+8,tip-14]],dtype=np.int32)
    cv2.polylines(frame,[pts],True,color,thickness,cv2.LINE_AA)
    cv2.circle(frame,(cx,tip),2,color,-1,cv2.LINE_AA)

def draw_possession_triangle(frame, bbox, color):
    """Filled downward-pointing triangle above player's head when they have the ball"""
    x1,y1,x2,y2 = bbox
    cx  = (x1+x2)//2
    tip = y1 - 10          # tip points down toward the player
    pts = np.array([[cx, tip],[cx-9, tip-16],[cx+9, tip-16]], dtype=np.int32)
    cv2.fillPoly(frame, [pts], color, cv2.LINE_AA)
    cv2.polylines(frame, [pts], True, (255,255,255), 1, cv2.LINE_AA)

def draw_ball_circle(frame, bbox, color, thickness=2):
    x1,y1,x2,y2 = bbox
    cx=(x1+x2)//2; cy=(y1+y2)//2
    r=max((x2-x1),(y2-y1))//2+4
    cv2.circle(frame,(cx,cy),r,color,thickness,cv2.LINE_AA)

def draw_label(frame, text, bbox, color):
    x1,y1,x2,y2 = bbox
    cx = (x1+x2)//2
    (tw,th),_ = cv2.getTextSize(text,FONT,FONT_SCALE,FONT_THICK)
    tx = cx-tw//2; ty = y1-18
    pad=3
    cv2.rectangle(frame,(tx-pad,ty-th-pad),(tx+tw+pad,ty+pad),(0,0,0),-1,cv2.LINE_AA)
    cv2.putText(frame,text,(tx,ty),FONT,FONT_SCALE,color,FONT_THICK,cv2.LINE_AA)

#  4. stats overlay
def draw_stats_overlay(frame, passes, possession_frames, total_frames, W, H):
    total_poss = possession_frames[0] + possession_frames[1]
    poss_pct   = [
        (possession_frames[0]/total_poss*100) if total_poss > 0 else 50,
        (possession_frames[1]/total_poss*100) if total_poss > 0 else 50,
    ]
    panel_w, panel_h = 340, 100
    px, py = W//2 - panel_w//2, 12
    overlay = frame.copy()
    cv2.rectangle(overlay,(px,py),(px+panel_w,py+panel_h),(0,0,0),-1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)

    t1_color = TEAM_BGR[0]; t2_color = TEAM_BGR[1]
    cv2.putText(frame,"TEAM 1",(px+10,py+22),FONT,0.5,t1_color,1,cv2.LINE_AA)
    cv2.putText(frame,"TEAM 2",(px+panel_w-70,py+22),FONT,0.5,t2_color,1,cv2.LINE_AA)

    p1_str = f"Passes: {passes[0]}"
    p2_str = f"Passes: {passes[1]}"
    cv2.putText(frame,p1_str,(px+10,py+44),FONT,0.4,(255,255,255),1,cv2.LINE_AA)
    (pw,_),_ = cv2.getTextSize(p2_str,FONT,0.4,1)
    cv2.putText(frame,p2_str,(px+panel_w-pw-10,py+44),FONT,0.4,(255,255,255),1,cv2.LINE_AA)

    bar_x, bar_y = px+10, py+58
    bar_w = panel_w-20; bar_h = 14
    t1_w  = int(bar_w * poss_pct[0]/100)
    cv2.rectangle(frame,(bar_x,bar_y),(bar_x+bar_w,bar_y+bar_h),(40,40,40),-1)
    cv2.rectangle(frame,(bar_x,bar_y),(bar_x+t1_w,bar_y+bar_h),t1_color,-1,cv2.LINE_AA)
    cv2.rectangle(frame,(bar_x+t1_w,bar_y),(bar_x+bar_w,bar_y+bar_h),t2_color,-1,cv2.LINE_AA)

    pct1 = f"{poss_pct[0]:.0f}%"
    pct2 = f"{poss_pct[1]:.0f}%"
    cv2.putText(frame,pct1,(bar_x+4,bar_y+11),FONT,0.38,(0,0,0),1,cv2.LINE_AA)
    (pw2,_),_ = cv2.getTextSize(pct2,FONT,0.38,1)
    cv2.putText(frame,pct2,(bar_x+bar_w-pw2-4,bar_y+11),FONT,0.38,(0,0,0),1,cv2.LINE_AA)
    cv2.putText(frame,"POSSESSION",(px+panel_w//2-38,bar_y+11),FONT,0.35,(220,220,220),1,cv2.LINE_AA)

    return frame

#  5. possession and pass logic
def ball_in_player_radius(ball_bbox, player_bbox, radius):
    ball_cx = (ball_bbox[0] + ball_bbox[2]) / 2
    ball_cy = (ball_bbox[1] + ball_bbox[3]) / 2
    p_cx    = (player_bbox[0] + player_bbox[2]) / 2
    p_cy    = player_bbox[3]   # bottom-center (feet)
    dist    = np.sqrt((ball_cx - p_cx)**2 + (ball_cy - p_cy)**2)
    return dist < radius, dist

#  6. tracker + state
tracker = sv.ByteTrack()

id_to_team          = {}
passes              = {0: 0, 1: 0}
poss_frames         = {0: 0, 1: 0}
last_possessor_id   = None
last_possessor_team = None
current_possessor   = None
poss_counter        = defaultdict(int)

# 7. render video
print("Step 4/4 — Rendering video with tracking + stats...")
cap    = cv2.VideoCapture(SOURCE_VIDEO)
fps    = cap.get(cv2.CAP_PROP_FPS)
W      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
raw_out= "/content/raw_tracked.mp4"
writer = cv2.VideoWriter(raw_out,cv2.VideoWriter_fourcc(*"mp4v"),fps,(W,H))

frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results   = model(frame, imgsz=1280, conf=CONF, verbose=False)[0]
    boxes     = results.boxes.xyxy.cpu().numpy()
    class_ids = results.boxes.cls.cpu().numpy().astype(int)
    confs     = results.boxes.conf.cpu().numpy()

    if len(boxes) == 0:
        writer.write(frame)
        frame_idx += 1
        continue

    ball_mask = class_ids == BALL_ID
    ball_boxes = boxes[ball_mask].astype(int)

    # split raw detections by class for team classification
    pl_mask  = class_ids == PLAYER_ID
    gk_mask  = class_ids == GOALKEEPER_ID
    ref_mask = class_ids == REFEREE_ID

    pl_boxes_raw  = boxes[pl_mask].astype(int)
    gk_boxes_raw  = boxes[gk_mask].astype(int)
    ref_boxes_raw = boxes[ref_mask].astype(int)

    # predict teams per-frame (same as reference code)
    pl_crops = []
    for b in pl_boxes_raw:
        x1,y1,x2,y2 = b
        crop = frame[y1:y2, x1:x2]
        pl_crops.append(
            Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            if crop.size > 0 else Image.new('RGB',(10,10))
        )
    pl_teams = predict_team(pl_crops) if pl_crops else np.array([], dtype=int)
    gk_teams = resolve_goalkeeper_team(pl_boxes_raw, pl_teams, gk_boxes_raw)

    #  track non-ball detections for possession/pass
    non_ball_boxes = boxes[~ball_mask]
    non_ball_cls   = class_ids[~ball_mask]
    non_ball_conf  = confs[~ball_mask]

    if len(non_ball_boxes) > 0:
        sv_dets = sv.Detections(
            xyxy      = non_ball_boxes,
            class_id  = non_ball_cls,
            confidence= non_ball_conf
        )
        sv_dets      = tracker.update_with_detections(sv_dets)
        tracked_boxes = sv_dets.xyxy.astype(int)
        tracked_ids   = sv_dets.tracker_id
        tracked_cls   = sv_dets.class_id
    else:
        tracked_boxes = np.empty((0,4),dtype=int)
        tracked_ids   = np.array([])
        tracked_cls   = np.array([])

    # cache team id per tracker id using per-frame classification
    new_player_mask = np.array([
        tracked_cls[i] in [PLAYER_ID, GOALKEEPER_ID] and tracked_ids[i] not in id_to_team
        for i in range(len(tracked_ids))
    ])
    if new_player_mask.any():
        new_crops = []
        for i in np.where(new_player_mask)[0]:
            x1,y1,x2,y2 = tracked_boxes[i]
            crop = frame[y1:y2, x1:x2]
            new_crops.append(
                Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
                if crop.size > 0 else Image.new('RGB',(10,10))
            )
        new_teams = predict_team(new_crops)
        for i, idx in enumerate(np.where(new_player_mask)[0]):
            id_to_team[tracked_ids[idx]] = int(new_teams[i])

    #  possession detection
    frame_possessor      = None
    frame_possessor_dist = float('inf')

    if len(ball_boxes) > 0:
        ball_bbox = ball_boxes[0]
        for i in range(len(tracked_boxes)):
            if tracked_cls[i] not in [PLAYER_ID, GOALKEEPER_ID]:
                continue
            in_radius, dist = ball_in_player_radius(ball_bbox, tracked_boxes[i], POSSESSION_RADIUS)
            if in_radius and dist < frame_possessor_dist:
                frame_possessor_dist = dist
                frame_possessor      = tracked_ids[i]

    if frame_possessor is not None:
        poss_counter[frame_possessor] += 1
        for k in list(poss_counter.keys()):
            if k != frame_possessor:
                poss_counter[k] = max(0, poss_counter[k] - 1)
                if poss_counter[k] == 0:
                    del poss_counter[k]
    else:
        for k in list(poss_counter.keys()):
            poss_counter[k] = max(0, poss_counter[k] - 1)
            if poss_counter[k] == 0:
                del poss_counter[k]

    confirmed_possessor = None
    for pid, cnt in poss_counter.items():
        if cnt >= POSSESSION_MIN_FRAMES:
            confirmed_possessor = pid

    if confirmed_possessor is not None:
        team = id_to_team.get(confirmed_possessor)
        if team is not None:
            poss_frames[team] += 1
            if (last_possessor_id is not None and
                last_possessor_id != confirmed_possessor and
                last_possessor_team == team):
                passes[team] += 1
            last_possessor_id   = confirmed_possessor
            last_possessor_team = team
        current_possessor = confirmed_possessor
    else:
        current_possessor = None

    #  draw ball
    for bbox in ball_boxes:
        draw_ball_circle(frame, bbox, BALL_BGR)
        draw_triangle(frame, bbox, BALL_BGR)
        draw_label(frame, "ball", bbox, BALL_BGR)

    #  draw players (per-frame team classification, no bbox)
    for bbox, team_id in zip(pl_boxes_raw, pl_teams):
        color   = TEAM_BGR[team_id]
        is_poss = any(
            tracked_ids[i] == current_possessor
            and abs(tracked_boxes[i][0]-bbox[0]) < 5
            for i in range(len(tracked_ids))
        )
        draw_ellipse(frame, bbox, color)
        draw_label(frame, f"T{team_id+1}", bbox, color)
        if is_poss:
            draw_possession_triangle(frame, bbox, color)

    #  draw goalkeepers
    for bbox, team_id in zip(gk_boxes_raw, gk_teams):
        color   = TEAM_BGR[team_id]
        is_poss = any(
            tracked_ids[i] == current_possessor
            and abs(tracked_boxes[i][0]-bbox[0]) < 5
            for i in range(len(tracked_ids))
        )
        draw_ellipse(frame, bbox, color, thickness=3)
        draw_label(frame, f"GK T{team_id+1}", bbox, color)
        if is_poss:
            draw_possession_triangle(frame, bbox, color)

    #  draw referees
    for bbox in ref_boxes_raw:
        draw_ellipse(frame, bbox, (128,128,128))
        draw_label(frame, "ref", bbox, (200,200,200))

    #  stats overlay
    frame = draw_stats_overlay(frame, passes, poss_frames, frame_idx+1, W, H)

    writer.write(frame)
    frame_idx += 1
    if frame_idx % 100 == 0:
        print(f"  {frame_idx}/{total} frames | passes T1:{passes[0]} T2:{passes[1]} | poss T1:{poss_frames[0]} T2:{poss_frames[1]}")

cap.release()
writer.release()
print("Compressing")

final_out = "/content/tracked_final.mp4"
subprocess.run([
    "ffmpeg","-y","-i",raw_out,
    "-vcodec","libx264","-crf","28",
    "-preset","fast","-movflags","+faststart",
    final_out
],check=True)

sz_in  = os.path.getsize(raw_out)  /(1024*1024)
sz_out = os.path.getsize(final_out)/(1024*1024)
print(f"Done! {sz_in:.1f}MB → {sz_out:.1f}MB")

from IPython.display import HTML
from base64 import b64encode
mp4      = open(final_out,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=800 controls><source src="{data_url}" type="video/mp4"></video>')